# Inference for categorical data

In August of 2012, news outlets ranging from the [Washington Post](https://www.washingtonpost.com/national/on-faith/poll-shows-atheism-on-the-rise-in-the-us/2012/08/13/90020fd6-e57d-11e1-9739-eef99c5fb285_story.html) to the [Huffington Post](https://www.huffpost.com/entry/atheism-rise-religiosity-decline-in-america_n_1777031) ran a story about the rise of atheism in America. The source for the story was a poll that asked people

> "Irrespective of whether you attend a place of worship or not, would you say you are a religious person, not a religious person or a convinced atheist?"

This type of question, which asks people to classify themselves in one way or another, is common in polling and generates categorical data. In this lab we take a look at the atheism survey and explore what's at play when making inference about population proportions using categorical data.

## The survey

Use the following link to access the press release for the poll, conducted by WIN-Gallup International:

> [Global Index of Religiosity and Atheism](https://drive.google.com/file/d/1bLauxSFDjdBZ0T2tOKqcDyj0XJ_LU819/view?usp=sharing)

Take a moment to review the report then address the following questions.

### Exercise 1

- In the first paragraph, several key findings are reported. Do these percentages appear to be *sample statistics* (derived from the data sample) or *population parameters*?
- The title of the report is "Global Index of Religiosity and Atheism." To generalize the report's findings to the global human population, what must we assume about the sampling method? Does that seem like a reasonable assumption?

## The data

Turn your attention to Table 6 (on pages 15 and 16 of document), which reports the sample size and response percentages for all 57 countries. While this is a useful format to summarize the data, we will base our analysis on the original data set of individual responses to the survey.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from plotnine import *
import io
import requests

df_url = 'https://raw.githubusercontent.com/akmand/datasets/master/openintro/atheism.csv'
url_content = requests.get(df_url, verify=False).content
religiosity_data = pd.read_csv(io.StringIO(url_content.decode('utf-8')))

In [ ]:
religiosity_data.head()

In [ ]:
religiosity_data.tail()

### Exercise 2

- What does each row of Table 6 from the report correspond to?
- What does each row of <code>religiosity_data</code> correspond to?

To investigate the link between these two ways of organizing this data, take a look at the estimated proportion of atheists in the United States. Towards the bottom of Table 6, we see that this is reported as 5%. We should be able to come to the same number using the `religiosity_data` dataframe.

### Exercise 3

- Fill in the code below to create a new dataframe called <code>us12</code> that contains only the rows in <code>religiosity_data</code> associated with respondents to the 2012 survey from the United States.

- Next, calculate the proportion of atheist responses.

- Does it agree with the percentage in Table 6? If not, why?

In [ ]:
us12 = religiosity_data[(religiosity_data['nationality'] == ???) & (religiosity_data['year'] == ???)] # Fill in the code!

us12.head() # Make sure you have the right data!

In [ ]:
# Calculate the proportion of `athiest` responses in the `us12` data.
# Hint: See Lab 1 for reference.



## Inference on proportions

As was hinted at in Exercise 1, Table 6 provides statistics, that is, calculations made from the sample of 51,927 people. What we'd like, though, is insight into the population parameters.

We answer the question

> "What proportion of people in your sample reported being atheists?"

with a statistic. However, the question

> "What proportion of people on earth would report being atheists"

is answered with an estimate of the population parameter.

The inferential tools for estimating population proportion are analogous to those studied in the last chapter: the confidence interval and the hypothesis test.

### Exercise 4

- Write out the conditions for inference to construct a 95% confidence interval for the proportion of atheists in the United States in 2012.
- Are you confident all conditions are met?

```
Hint: See Section 6.1.1 of the textbook if you're stuck.
```

If the conditions for inference are reasonable, we can either calculate the standard error and construct the interval by hand. Or we can use [`scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html), a Python library of useful statistical functions offered by `SciPy`.

Run the code below to create a 95% confidence interval for the proportion of atheists in the US in 2012 using the `confidence_interval()` function.

In [ ]:
# Run this code to define the `confidence_interval()` function.

import numpy as np
from scipy.stats import norm

def confidence_interval(conf_lvl, data):
  conf_lvl = conf_lvl
  z_value = norm.ppf((1-(1-conf_lvl)/2))  # the Z value for the specified confidence level
  prbs = data.response.value_counts(normalize = True)  # the probabilities for the response of atheist and non-atheist
  se = np.sqrt(prbs.prod()/len(data))  # the standard error

  # construct a confidence interval for the proportion of atheists in the United States in 2012
  ci1 = prbs['atheist'] - z_value * se
  ci2 = prbs['atheist'] + z_value * se

  # print the results
  print(f'Number of successes (atheist) = {len(data)*prbs[1]}')
  print(f'Number of failures (non-atheist) = {len(data)*prbs[0]}')
  print(f'z_* value = {z_value}')
  print(f'Standard error = {se}')
  print(f'{int(conf_lvl*100)}% confidence interval = {ci1, ci2}')

In [ ]:
# Run this code to compute the desired confidence interval.

confidence_interval(conf_lvl = 0.95, data = us12)

Although formal confidence intervals and hypothesis tests don't show up in the report, suggestions of inference appear at top of page 8.

> "In general, the error margin for surveys of this kind is ± 3-5% at 95% confidence."

### Exercise 5

- Based on the code's output, what is the margin of error for the estimate of the proportion of the proportion of atheists in US in 2012? Round to four decimal places.
- How does this compare to the reports statement of a 3–5% margin of error?

## How does the proportion affect the margin of error?

Imagine you've set out to survey 1000 people on two questions

> "Are you female?"

and

> "Are you left-handed?"

Since both of these sample proportions were calculated from the same sample size, they should have the same margin of error, right? Wrong! While the margin of error does change with sample size, it is also affected by the proportion.

Think back to the formula for the standard error:

$$SE = \sqrt{\frac{p(1−p)}{n}}.$$

This is then used in the formula for the margin of error for a 95% confidence interval:

$$ ME = {1.96} \times {SE} = 1.96 \times \sqrt{\frac{p(1−p)}{n}}. $$

Since the population proportion ${p}$ is in this ME formula, it should make sense that the margin of error is in some way dependent on the population proportion. We can visualize this relationship by creating a plot of ${ME}$ vs. ${p}$.

Run the code below to plot the relationship between $p$ and $ME$.

In [ ]:
import numpy as np

n = 1000

(
    ggplot(pd.DataFrame({"p": [0, 1]})) +
    aes(x = "p") +
    stat_function(fun=lambda p: 1.96 * np.sqrt(p * (1 - p)/n)) +
    labs(x = "Population Proportion (p)", y = "Margin of Error")
)

### Exercise 6

- For what value of $p$ does the margin of error appear largest?
- Suppose you're hired by the local government to estimate the proportion of residents that attend a religious service on a weekly basis. According to the guidelines, the estimate must have a margin of error no greater than 1% with 95% confidence. You have no idea what to expect for ${p}$. How many people would you have to sample to *ensure* that you are within the guidelines?

```
Hint: See Section 6.1.5 if you're stuck.
```

## Success-failure condition

The textbook emphasizes that you must always check conditions before making inference. For inference on proportions, the sample proportion can be assumed to be nearly normal if it is based upon a random sample of independent observations and if both ${np}$ ≥ ${10}$ and ${n}$(${1 - p}$) ≥ ${10}$. This guideline is easy enough to follow, but it makes one wonder

> What's so special about the number 10?

The short answer is: nothing!

You could argue that we would be fine with 9 or that we really should be using 11 — in fact, many modern references insist upon 15. What is the "best" value for such a guideline is, at least to some degree, arbitrary. However, when ${np}$ and ${n}$(${1 − p}$) reaches 10 the sampling distribution is sufficiently normal to use confidence intervals and hypothesis tests that are based on that approximation.

We can investigate the interplay between ${n}$ and ${p}$ and the shape of the sampling distribution by using simulations. To start off, we simulate the process of drawing 5000 samples of size 1040 from a population with a true atheist proportion of 0.1. For each of the 5000 samples we compute ${\hat{p}}$ and then plot a histogram to visualize their distribution.

In [ ]:
p = 0.1
n = 1040
p_hats = np.zeros(5000)

for i in range(5000):
    samp = np.random.choice(['atheist', 'non_atheist'], size = n, replace = True, p = [p, 1-p])
    p_hats[i] = sum(samp == 'atheist')/n

(
    ggplot(pd.DataFrame(p_hats, columns = ['p_hats'])) +
    aes(x = "p_hats") +
    geom_histogram(binwidth = 0.005) +
    labs(title = f'p = {p}, n = {n}')
)

These commands build up the sampling distribution of ${\hat{p}}$ using a `for` loop. You can read the sampling procedure for the first line of code inside the `for` loop as

> "Take a sample of size ${n}$ with replacement from the choices of atheist and non-atheist with probabilities  ${p}$  and  ${1 - p}$ , respectively."

The second line in the loop says

> "Calculate the proportion of atheists in this sample and record this value."

The loop allows us to repeat this process 5,000 times to build a good representation of the sampling distribution.

### Exercise 7

Describe the sampling distribution of sample proportions at  ${n = 1040}$ and  ${p = 0.1}$. Be sure to note the center, spread, and shape.

```
Hint: You can run the code below for a numerical summary of the sampling distribution.
```

In [ ]:
# This yields a numerical summary of p_hats.

pd.DataFrame(p_hats).describe()

### Exercise 8

- Repeat the above simulation three more times but with modified sample sizes and proportions as follows.
  - ${p = 0.1}$ and ${n = 400}$,
  - ${p = 0.02}$ and ${n = 1040}$,
  - ${p = 0.02}$ and ${n = 400}$.
- Describe the three new sampling distributions.
- Based on these limited plots, how does ${n}$ appear to affect the sampling distribution of ${\hat{p}}$? How does ${p}$ affect the sampling distribution?

```
Hint: See Section 5.1.5 of the textbook if you're stuck.
```

In [ ]:
p = 0.1
n = 400
p_hats = np.zeros(5000)

for i in range(5000):
    samp = np.random.choice(['atheist', 'non_atheist'], size = n, replace = True, p = [p, 1-p])
    p_hats[i] = sum(samp == 'atheist')/n

(
    ggplot(pd.DataFrame(p_hats, columns = ['p_hats'])) +
    aes(x = "p_hats") +
    geom_histogram(binwidth = 0.005) +
    labs(title = f'p = {p}, n = {n}')
)

In [ ]:
p = 0.02
n = 1040
p_hats = np.zeros(5000)

for i in range(5000):
    samp = np.random.choice(['atheist', 'non_atheist'], size = n, replace = True, p = [p, 1-p])
    p_hats[i] = sum(samp == 'atheist')/n

(
    ggplot(pd.DataFrame(p_hats, columns = ['p_hats'])) +
    aes(x = "p_hats") +
    geom_histogram(binwidth = 0.005) +
    labs(title = f'p = {p}, n = {n}')
)

In [ ]:
p = 0.02
n = 400
p_hats = np.zeros(5000)

for i in range(5000):
    samp = np.random.choice(['atheist', 'non_atheist'], size = n, replace = True, p = [p, 1-p])
    p_hats[i] = sum(samp == 'atheist')/n

(
    ggplot(pd.DataFrame(p_hats, columns = ['p_hats'])) +
    aes(x = "p_hats") +
    geom_histogram(binwidth = 0.005) +
    labs(title = f'p = {p}, n = {n}')
)

If you'd like to investigate more examples, run the code below to define `sample_distribution()`, which is a function that plots a simulated sampling distribution for given $n$ and $p$ values. Then you can call `sample_distribution(???,???)` to plot simulated sampling distributions for a variety of $n$ and $p$ values.

In [ ]:
def sample_distribution(p, n):
  p = p
  n = n
  p_hats = np.zeros(5000)

  for i in range(5000):
      samp = np.random.choice(['atheist', 'non_atheist'], size = n, replace = True, p = [p, 1-p])
      p_hats[i] = sum(samp == 'atheist')/n

  plot = (
      ggplot(pd.DataFrame(p_hats, columns = ['p_hats'])) +
      aes(x = "p_hats") +
      geom_histogram(binwidth = 0.005) +
      labs(title = f'p = {p}, n = {n}')
  )

  return plot

In [ ]:
sample_distribution(p = ???, n = ???)

### Exercise 9

If you refer to Table 6, you'll find that Australia has a sample proportion of 0.1 on a sample size of 1040 and that Ecuador has a sample proportion of 0.02 on 400 subjects.

Let's suppose for this exercise that these point estimates are actually the truth. Then, given the shape of their respective sampling distributions (visualized in the previous exercise), do you think it is sensible to proceed with inference and report margins of error, as the reports does?

```
Hint: Do all of the simulated sampling distributions look normal? Do
they all satisfy the conditions for inference?
```

## Testing for change in the atheism index

We'd now like to ask ourselves

> Is there convincing evidence that the United States has seen a change in its atheism index between 2005 and 2012?

To do this, we'll proceed using a confidence interval *as well as* a hypothesis test on the difference of proportions as in Section 6.2.

### Exercise 10

We've already created the `us12` data set and calculated a 95% confidence interval for the proportion of atheists in the US in 2012 (see Exercises 4 and 5).

- Now, create a data set `us05` containing only the rows in <code>religiosity_data</code> associated with respondents to the 2005 survey from the United States.
- What is the difference in proportions of atheist responses from 2005 to 2012?
- Determine whether it is reasonable to proceed with statistical inference given the data we have in `us05` and `us12`.
- If so, write out appropriate null and alternative hypotheses.

```
Hint: We already know the n and p values from `us12`
(see Exercise 3 and/or Table 6). Determine them for
`us05` as well (round to 4 decimal places), find their
difference, and make an appropriate conclusion about
inference using the success-failure condition on each
data set.
```

In [ ]:
# Create `us05`

us05 = religiosity_data[(religiosity_data['nationality'] == ???) & (religiosity_data['year'] == ???)] # Fill in the code!

us05.head() # Make sure you have the right data!

In [ ]:
# Find the proportion of `atheist` responses in `us05`
# Hint: See Exercise 3 of this Lab.



In [ ]:
# Find the difference in proportions



### Exercise 11

- If statistical inference is reasonable, use the `confidence_interval_difference()` function defined below to find a 95% confidence interval for the difference of proportions of atheists in the US between 2005 and 2012.
- Does the resulting confidence interval contain 0? Using this observation, draw an appropriate conclusion regarding whether there is convincing evidence that the United States has seen a change in its atheism index between 2005 and 2012.

In [ ]:
# This code defines the `confidence_interval_difference()` function.
# It depends on a confidence level and the input of two data sets.

import numpy as np
from scipy.stats import norm

def confidence_interval_difference(conf_lvl, data1, data2):
  conf_lvl = conf_lvl
  z_value = norm.ppf((1-(1-conf_lvl)/2))  # the Z value for the specified confidence level
  prbs1 = data1.response.value_counts(normalize = True)  # the probabilities for the response of atheist and non-atheist
  prbs2 = data2.response.value_counts(normalize = True)
  se = np.sqrt((prbs1.prod()/len(data1)) + (prbs2.prod()/len(data2)))  # the standard error

  # construct a confidence interval for the proportion of atheists in the United States in 2012
  ci1 = (prbs1['atheist'] - prbs2['atheist']) - z_value * se
  ci2 = (prbs1['atheist'] - prbs2['atheist']) + z_value * se

  # print the results
  print(f"Proportion 1 = {prbs1['atheist']}")
  print(f"Proportion 2 = {prbs2['atheist']}")
  print(f"Difference in proportions = {prbs1['atheist'] - prbs2['atheist']}")
  print(f'z_* value = {z_value}')
  print(f'Standard error = {se}')
  print(f'{int(conf_lvl*100)}% confidence interval = {ci1, ci2}')

In [ ]:
# If reasonable, use `confidence_interval()` as described above.
# The first input is the desired confidence level.
# The second and third inputs are the datasets.

confidence_interval_difference(conf_lvl = ???, data1 = ???, data2 = ???)

### Exercise 12

We'll use a p-value to examine our hypotheses in this Exercise, as opposed to a confidence interval as done in the previous Exercise.

- Calculate the **pooled proportion** (see page 220 of the textbook). You can round to four decimal places.
- If statistical inference is reasonable, use the `hypothesis_test_difference()` function to find a p-value for our hypothesis test.
- Using a significance level of $\alpha = 0.05$, draw an appropriate conclusion from the reported p-value.
- Is this the same conclusion you came to in Exercise 11?

In [ ]:
# Identify the proportions in 2005 (p_05) and 2012 (p_12).

p_05 =
p_12 =

# Compute the pooled proportion from p_05 and p_12.

p_pooled =

# The code below will print the proportions for later reference.

print(f"Proportion from 2005: {p_05}")
print(f"Proportion from 2012: {p_12}")
print(f"Difference of proportions: {p_12 - p_05}")
print(f"Pooled proportion: {p_pooled}")

In [ ]:
# Check the success-failure conditions using the pooled proportion.

print(f"Successes in 2005: {len(us05)*p_pooled}")
print(f"Failures in 2005: {len(us05)*(1-p_pooled)}")
print(f"Successes in 2012: {len(us12)*p_pooled}")
print(f"Failures in 2012: {len(us12)*(1-p_pooled)}")

In [ ]:
# This code defines the `hypothesis_test_difference()` function.
# Given the difference of proportions, the pooled proption, and the null value
# of the proportion, it will report back the corresponding standard error,
# Z-score, and p-value for a hypothesis test.

import numpy as np
from scipy.stats import norm

def hypothesis_test_difference(p_diff, p_pooled, p_null):
  p_diff = p_diff
  p_pooled = p_pooled
  p_null = p_null

  se = np.sqrt(((p_pooled * (1 - p_pooled))/1002) + ((p_pooled * (1 - p_pooled))/1002)) # Find the standard error using the pooled proportion
  z_score = (p_diff - p_null)/se # Find the z-score
  p_value = 2 * (1 - norm.cdf(abs(z_score))) # Find the p-value

  # Print the results
  print(f'Difference in proportions = {p_diff}')
  print(f'Standard error = {se}')
  print(f'Z-score = {z_score}')
  print(f'p-value = {p_value}')

In [ ]:
# If reasonable, use `hypothesis_test_difference()` as described above.
# The first input is the difference in proportions.
# The second input is the pooled proportion.
# The third input is the proportion's value in the null hypothesis.

hypothesis_test_difference(p_diff = ???, p_pooled = ???, p_null = ???)

---

This lab was adapted by Timothy L. Clark, derivative of [OpenIntro Statistics by Diez, Çetinkaya-Rundel, and Barr](https://www.openintro.org/book/os/), released under [Creative Commons BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/deed.en) license.